# Book Topic Modeling and Financial Entity Extraction
**Cameron Batts**

Python · NLP · Gensim · LDA · spaCy · Dependency Parsing · NER

## Overview
Explore two complementary NLP workflows: discover themes in 16,559 book summaries with Latent Dirichlet Allocation, and inspect named entities in 4,846 financial-news sentences. Compare topic counts and training passes, interpret document mixtures, and examine entity extraction quality.

## Business context
Topic models can support collection browsing and content organization; entity extraction can structure organizations, locations, and monetary information for document review. These are potential applications, not measured production outcomes.

## Data sources
Book summaries: CMU Book Summary Dataset, https://www.cs.cmu.edu/~dbamman/booksummaries.html . Financial news: Malo et al. (2014), *Good debt or bad debt: Detecting semantic orientations in economic texts*, https://doi.org/10.1002/asi.23062 . Required source attribution is retained. Local CSV copies are required; raw datasets and person-name examples are not distributed in this project.

## Execution status
Selected outputs are retained from the supplied completed notebook. The cleaned code is syntax-checked but has not been rerun. Revised span-level NER and dependency examples are clearly marked as unexecuted extensions.

## Validation update - October 22, 2026
The full workflow executed on 16,559 book summaries and 4,846 financial sentences, including revised raw-text full-span NER.

| Configuration | Rerun training-corpus fit score |
| --- | ---: |
| 5topics_2passes | 431.4384 |
| 10topics_2passes | 548.1752 |
| 10topics_3passes | 539.8176 |
| 10topics_5passes | 532.5633 |

Scores retain the source's `2 ** (-log_perplexity)` calculation and are not held-out evaluation. Aggregate raw-span counts are recorded without redistributing source sentences, names or entity examples. No topic-quality, NER-accuracy or business-outcome benchmark was added.


In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import nltk

def ensure_nltk_resource(location, package):
    """Reuse local resources; download only when absent."""
    try:
        nltk.data.find(location)
    except LookupError:
        if not nltk.download(package, quiet=True):
            raise RuntimeError(f'Install the NLTK {package} resource locally, then rerun.')

from gensim.models import LdaModel
from gensim.corpora import Dictionary
import gensim.parsing.preprocessing as preprocessing
from gensim.utils import simple_preprocess

ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
ensure_nltk_resource('corpora/wordnet', 'wordnet')
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
book_df = pd.read_csv(root / 'data' / 'book.csv', usecols=['title', 'summary_name_removed'])
book_df.dropna(subset=['summary_name_removed'], inplace=True)
print('Book summaries:', len(book_df))


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Book preprocessing
Use the supplied name-reduced summary column, lowercase text, remove punctuation, digits, extra whitespace and stopwords, then apply POS-aware lemmatization. Automated name removal is imperfect and can leave residual proper names in source text; raw examples are omitted from saved outputs.

In [2]:
def lemmatize_text(token_list, wnl):
  """
  This function tags the pos of the input tokens,
  and lemmatize the tokens.
  """
  # POS tag each word
  for word, tag in nltk.pos_tag(token_list):
    # Mapping the pos tags to the types supported by wnl
    if tag.startswith("NN"):
      yield wnl.lemmatize(word, pos='n')
    elif tag.startswith('VB'):
      yield wnl.lemmatize(word, pos='v')
    elif tag.startswith('JJ'):
      yield wnl.lemmatize(word, pos='a')
    elif tag.startswith('RB'):
      yield wnl.lemmatize(word, pos='r')
    else:
      yield wnl.lemmatize(word)

# lower letters, remove punctuations, digits, multiple whitespaces, and stopwords
CUSTOM_FILTERS = [lambda x: x.lower(), preprocessing.strip_punctuation, preprocessing.strip_numeric, preprocessing.strip_multiple_whitespaces, preprocessing.remove_stopwords]
book_df["processed_summary"] = book_df["summary_name_removed"].apply(lambda x: preprocessing.preprocess_string(x, CUSTOM_FILTERS))

# lemmatize the tokens
wnl = nltk.WordNetLemmatizer()
book_df["processed_summary"] = book_df["processed_summary"].apply(lambda x: simple_preprocess(" ".join(lemmatize_text(x, wnl))))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Dictionary and models
Remove the 50 most frequent dictionary tokens. Train 5- and 10-topic LDA models with random_state=30, iterations=50 and two passes.

In [3]:
# create a dictionary from the processed summaries
summary_list = book_df["processed_summary"].tolist()
book_dictionary = Dictionary(summary_list)
# filter top 50 most frequent tokens
book_dictionary.filter_n_most_frequent(50)
# build the corpus for LDA training
book_corpus = [book_dictionary.doc2bow(text) for text in summary_list]

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [4]:
lda_10topics = LdaModel(corpus = book_corpus, num_topics = 10, id2word = book_dictionary, passes = 2, iterations = 50, random_state = 30)

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [5]:
lda_5topics = LdaModel(corpus=book_corpus,num_topics=5,id2word=book_dictionary,passes=2,iterations=50,random_state=30)

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Corpus-fit comparison
The original calculation applies `2 ** (-log_perplexity)` to Gensim's approximate per-word variational bound on the same corpus used for training. The values below are retained from that calculation. They are not held-out perplexity, classification accuracy, or a guarantee of better human-interpretable topics.

In [6]:
perplexity_5 = np.exp2(-lda_5topics.log_perplexity(book_corpus))
perplexity_10 = np.exp2(-lda_10topics.log_perplexity(book_corpus))

print("Perplexity for lda_5topics:", perplexity_5)
print("Perplexity for lda_10topics:", perplexity_10)

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [7]:
lda_3passes = LdaModel(corpus=book_corpus,num_topics=10,id2word=book_dictionary,passes=3,iterations=50,random_state=30)
lda_5passes = LdaModel(corpus=book_corpus,num_topics=10,id2word=book_dictionary,passes=5,iterations=50,random_state=30)

print("Perplexity for lda_3passes:", np.exp2(-lda_3passes.log_perplexity(book_corpus)))
print("Perplexity for lda_5passes:", np.exp2(-lda_5passes.log_perplexity(book_corpus)))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


The five-topic configuration has the lowest reported corpus-fit score (431.44). For ten topics, increasing passes from two to five reduces the score from 548.17 to 532.56. This supports improved fit under the reported calculation, not an overall model-quality ranking.

Topic count should also reflect interpretability, topic coherence, stability and browsing needs. Five broad topics can suit an overview; ten topics can provide finer distinctions. A held-out split would strengthen evaluation.

## Broad topic inspection
The original five-topic output suggests family/school, narrative/history, science fiction/space, fantasy/conflict, and war/government themes. These names are human interpretations, not learned genre labels.

In [8]:
lda_5topics.print_topics(num_words=10)

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [9]:
# Inspect the finer model after running; source-data names can appear in generated topics.
lda_10topics.print_topics(num_words=10)


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Document mixtures
Use positional corpus entries for three illustrative documents. The original top reported topic weights were 0.4063 for topic 0 at position 565, 0.5200 for topic 0 at position 480, and 0.3998 for topic 8 at position 532. These documents are part of the training corpus, not independent test cases. Topic weights describe mixtures rather than classification accuracy. Default minimum-probability filtering can omit small topic weights.

In [10]:
lda_10topics.get_document_topics(book_corpus[565])

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [11]:
lda_10topics.get_document_topics(book_corpus[480])

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [12]:
lda_10topics.get_document_topics(book_corpus[532])

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Interpretation
The original review found that family and school vocabulary could dominate a fantasy-oriented document. Topic models group co-occurring words and do not necessarily reproduce human genre categories. LDA estimates nonnegative document-topic mixtures, whereas latent semantic indexing uses a lower-rank algebraic representation; either approach needs task-specific evaluation. Topics alone should not be assumed to represent positive, neutral or negative sentiment.

## Dependency parsing: new illustrative example
Install the spaCy language model separately. The following neutral example replaces personal names and course-specific text; no original output is claimed for this revised sentence.

In [13]:
import spacy
from spacy import displacy
nlp = spacy.load('en_core_web_md')
doc = nlp('The analyst reviewed the report before the meeting.')
parse_table = pd.DataFrame([(t.text, t.pos_, t.tag_, t.ent_type_, t.dep_) for t in doc],
                          columns=['TOKEN', 'POS', 'TAG', 'ENTITYTYPE', 'DEPENDENCY'])
print(parse_table)
displacy.render(doc, style='dep', jupyter=True, options={'distance': 100})


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Financial-news baseline
Read the local headerless Latin-1 CSV. The original baseline strips punctuation, numbers, stopwords and single-letter tokens before spaCy. This can erase amounts and disrupt names and entity boundaries. It then counts tokens tagged MONEY, GPE or ORG, rather than complete entity spans.

In [14]:
news_df = pd.read_csv(root / 'data' / 'all-data.csv', header=None, usecols=[1], names=['content'], encoding='latin-1')
print('News sentences:', len(news_df))


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [15]:
# We next do several necessary preprocessing steps on the dataset, which removes punctuation, digits, stopwords and single-letter words
# You can run the code without any edits needed.
CUSTOM_FILTERS = [preprocessing.strip_punctuation, preprocessing.strip_numeric, preprocessing.remove_stopwords, lambda x: preprocessing.strip_short(x, minsize=2)]
news_df["processed_content"] = news_df["content"].apply(lambda x: ' '.join(preprocessing.preprocess_string(x, CUSTOM_FILTERS)))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [16]:
news_df["nlp"] = news_df["processed_content"].apply(nlp)

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [17]:
# find the "MONEY" entities in each sentence
news_df["money"] = news_df['nlp'].apply(lambda doc: [token.text for token in doc if token.ent_type_ == "MONEY"])
# return the top-10 frequent entities
news_df["money"].explode().value_counts()[:10]

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [18]:
news_df["GPE"] = news_df['nlp'].apply(lambda doc: [token.text for token in doc if token.ent_type_ == "GPE"])
news_df["GPE"].explode().value_counts()[:10]


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [19]:
news_df["ORG"] = news_df['nlp'].apply(lambda doc: [token.text for token in doc if token.ent_type_ == "ORG"])
news_df["ORG"].explode().value_counts()[:10]

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Entity-quality findings
The baseline's MONEY-token list includes mln (168), million (162), and EUR (149), which are fragments or units rather than complete monetary amounts. GPE counts include Finland (261) but also mn (125); ORG counts include Nokia (120), EUR (137), and Finland (64). These examples expose tagging errors and preprocessing artifacts, not validated entity statistics. Some labels depend on sentence context.

Because numbers were stripped and spans were split into tokens, these results cannot reliably answer questions about monetary amounts or entity mention counts.

## Extension: preserve text and count complete spans
This revised workflow runs spaCy on the original sentences and counts complete `doc.ents` spans for selected categories. It has not been executed during portfolio preparation. Manual validation, normalization and labeled evaluation are still needed.

In [20]:
span_counts = {label: Counter() for label in ['MONEY', 'GPE', 'ORG']}
for document in nlp.pipe(news_df['content'].astype(str), batch_size=64):
    for entity in document.ents:
        if entity.label_ in span_counts:
            span_counts[entity.label_][entity.text] += 1
for label, counts in span_counts.items():
    print(label, counts.most_common(10))


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Limitations and next steps
- Evaluate LDA on held-out documents and inspect coherence and topic stability; two passes may be insufficient.
- Name-reduced summaries can retain names; removing the most frequent tokens may also remove useful signals.
- Inspect alignment when changing filtering or indexing of book rows.
- Validate NER on original text with complete spans and manually reviewed examples.
- No genre-classification accuracy, NER precision/recall or business impact was measured.

## Author
Cameron Batts